# Music Annotation Model Evaluation Suite

**Runtime:** T4 GPU. Select **Runtime > Run all** after editing the configuration cell.

This notebook does not retrain or modify models or datasets. It evaluates one or more YOLO segmentation checkpoints using:

- confidence sweeps on every Gold Standard Testing page;
- visual contact sheets for each threshold;
- a hard-negative false-positive leaderboard;
- a broad-mask risk index;
- model-to-model comparisons;
- optional Ultralytics validation metrics;
- CSV, JSON, charts, overlays and automatic recommendations.

The most useful outputs are:

- `gst_threshold_sweep.csv`
- `hard_negative_false_positive_leaderboard.csv`
- `model_comparison.csv`
- `instance_predictions.csv`
- GST and hard-negative contact sheets
- `evaluation_summary.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ultralytics pandas matplotlib

## Configuration

Update the checkpoint paths and folders. `MODEL_PATHS` can contain one model or several models for direct comparison.

Use `NORMALISATION="original"` for the first controlled comparison. Use `"all"` later to compare original, background-normalised and CLAHE inputs.

In [ ]:
from pathlib import Path

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')

MODEL_PATHS={
    # Use short, meaningful names. Add or remove models as needed.
    'current': ROOT/'training_runs'/'20261005_151805_yolo11s-seg'/'weights'/'best.pt',
    'baseline': ROOT/'baseline_models'/'20261005_yolo11n_HardNegative_Baseline_v2'/'best.pt',
}

GST_DIR=ROOT/'gold_standard_testing'
HARD_NEGATIVE_DIR=ROOT/'hard_negative_testing'
DATASET_YAML=ROOT/'dataset_v3.yaml'
OUTPUT_ROOT=ROOT/'evaluation_runs'

CONFIDENCE_THRESHOLDS=[0.10,0.20,0.30,0.40]
TILE_SIZE=640
TILE_STRIDE=320
MODEL_IMAGE_SIZE=640
DEVICE='0'
IOU_THRESHOLD=0.70
MINIMUM_OUTPUT_COMPONENT_AREA=50

# original, bgnorm, clahe, or all
NORMALISATION='original'
BACKGROUND_BLUR_SIGMA=75.0
CLAHE_CLIP_LIMIT=2.0
MAXIMUM_CONTACT_SHEET_PAGES=30

RUN_VALIDATION_METRICS=True

print('Models:')
for name,path in MODEL_PATHS.items(): print(' ',name,'->',path)
print('GST directory:',GST_DIR)
print('Hard-negative directory:',HARD_NEGATIVE_DIR)
print('Thresholds:',CONFIDENCE_THRESHOLDS)

## Imports, validation and evaluation functions

In [ ]:
import cv2
import json
import hashlib
import re
from datetime import datetime, timezone
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
import argparse

SUPPORTED_IMAGES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}


def image_files(folder: Optional[Path]) -> List[Path]:
    if folder is None:
        return []
    if not folder.exists():
        raise FileNotFoundError(f"Image directory not found: {folder}")
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_IMAGES
    )


def safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", text).strip("_")


def tile_positions(length: int, tile_size: int, stride: int) -> List[int]:
    if length <= tile_size:
        return [0]
    positions = list(range(0, length - tile_size + 1, stride))
    final = length - tile_size
    if positions[-1] != final:
        positions.append(final)
    return positions


def normalise_background(gray: np.ndarray, sigma: float) -> np.ndarray:
    background = cv2.GaussianBlur(gray, (0, 0), sigma)
    background = np.maximum(background, 1)
    return cv2.divide(gray, background, scale=255)


def apply_clahe(gray: np.ndarray, clip_limit: float) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
    return clahe.apply(gray)


def image_variants(
    gray: np.ndarray,
    mode: str,
    background_sigma: float,
    clahe_clip_limit: float,
) -> Dict[str, np.ndarray]:
    if mode == "original":
        return {"original": gray}
    background = normalise_background(gray, background_sigma)
    if mode == "bgnorm":
        return {"bgnorm": background}
    clahe = apply_clahe(background, clahe_clip_limit)
    if mode == "clahe":
        return {"clahe": clahe}
    return {"original": gray, "bgnorm": background, "clahe": clahe}


def remove_small_components(mask: np.ndarray, minimum_area: int) -> np.ndarray:
    if minimum_area <= 0 or not np.any(mask):
        return mask
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    output = np.zeros_like(mask, dtype=bool)
    for component in range(1, n):
        if stats[component, cv2.CC_STAT_AREA] >= minimum_area:
            output[labels == component] = True
    return output


def tiled_predict(
    model: YOLO,
    gray: np.ndarray,
    threshold: float,
    tile_size: int,
    stride: int,
    imgsz: int,
    device: str,
    iou: float,
    minimum_component_area: int,
) -> Tuple[np.ndarray, np.ndarray, List[dict]]:
    colour = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    height, width = gray.shape
    padded_height = max(height, tile_size)
    padded_width = max(width, tile_size)
    canvas = np.full((padded_height, padded_width, 3), 255, np.uint8)
    canvas[:height, :width] = colour
    confidence_map = np.zeros((padded_height, padded_width), np.float32)
    instances: List[dict] = []

    for y in tile_positions(padded_height, tile_size, stride):
        for x in tile_positions(padded_width, tile_size, stride):
            tile = canvas[y:y + tile_size, x:x + tile_size]
            result = model.predict(
                tile,
                conf=threshold,
                iou=iou,
                imgsz=imgsz,
                device=device,
                verbose=False,
                retina_masks=True,
            )[0]
            if result.masks is None:
                continue
            masks = result.masks.data.cpu().numpy()
            confidences = (
                result.boxes.conf.cpu().numpy()
                if result.boxes is not None
                else np.ones(len(masks), dtype=np.float32)
            )
            for mask, confidence in zip(masks, confidences):
                resized = cv2.resize(
                    mask,
                    (tile_size, tile_size),
                    interpolation=cv2.INTER_NEAREST,
                ) > 0.5
                pixels = int(resized.sum())
                coverage = 100.0 * pixels / (tile_size * tile_size)
                instances.append(
                    {
                        "tile_x": int(x),
                        "tile_y": int(y),
                        "confidence": float(confidence),
                        "mask_pixels": pixels,
                        "tile_coverage_pct": coverage,
                    }
                )
                region = confidence_map[y:y + tile_size, x:x + tile_size]
                region[resized & (confidence > region)] = confidence

    confidence_map = confidence_map[:height, :width]
    selected = confidence_map >= threshold
    selected = remove_small_components(selected, minimum_component_area)
    overlay = colour.copy()
    overlay[selected] = (0, 0, 255)
    output = cv2.addWeighted(colour, 0.68, overlay, 0.32, 0)
    return output, confidence_map, instances


def summarise_prediction(
    page: str,
    group: str,
    model_name: str,
    variant: str,
    threshold: float,
    gray: np.ndarray,
    confidence_map: np.ndarray,
    instances: List[dict],
    prediction_path: Path,
) -> dict:
    selected = confidence_map >= threshold
    predicted_pixels = int(selected.sum())
    page_pixels = int(gray.size)
    instance_confidences = [x["confidence"] for x in instances]
    instance_coverages = [x["tile_coverage_pct"] for x in instances]
    selected_confidences = confidence_map[selected]
    predicted_page_pct = 100.0 * predicted_pixels / page_pixels
    largest_instance_pct = max(instance_coverages, default=0.0)
    risk_index = predicted_page_pct * largest_instance_pct
    return {
        "group": group,
        "page": page,
        "model": model_name,
        "variant": variant,
        "threshold": threshold,
        "instances": len(instances),
        "predicted_pixels": predicted_pixels,
        "page_pixels": page_pixels,
        "predicted_page_pct": predicted_page_pct,
        "mean_pixel_confidence": float(selected_confidences.mean()) if predicted_pixels else 0.0,
        "maximum_confidence": float(confidence_map.max()),
        "median_instance_confidence": float(np.median(instance_confidences)) if instance_confidences else 0.0,
        "p95_instance_confidence": float(np.percentile(instance_confidences, 95)) if instance_confidences else 0.0,
        "largest_instance_tile_pct": largest_instance_pct,
        "risk_index": risk_index,
        "prediction_file": str(prediction_path),
    }


def save_contact_sheet(
    records: pd.DataFrame,
    title: str,
    output_path: Path,
    maximum_pages: int,
) -> None:
    if records.empty:
        return
    page_names = list(dict.fromkeys(records["page"].tolist()))[:maximum_pages]
    thresholds = sorted(records["threshold"].unique())
    rows = len(page_names)
    columns = len(thresholds)
    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(columns * 6, max(1, rows) * 6),
        squeeze=False,
    )
    for row_index, page in enumerate(page_names):
        for column_index, threshold in enumerate(thresholds):
            subset = records[
                (records["page"] == page)
                & (records["threshold"] == threshold)
            ]
            axis = axes[row_index][column_index]
            if subset.empty:
                axis.axis("off")
                continue
            record = subset.iloc[0]
            image = cv2.imread(record["prediction_file"])
            if image is not None:
                axis.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            axis.set_title(
                f"{page} | conf {threshold:.2f}\n"
                f"page={record['predicted_page_pct']:.2f}% "
                f"risk={record['risk_index']:.2f}"
            )
            axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=140, bbox_inches="tight")
    plt.close(fig)


def save_threshold_chart(df: pd.DataFrame, output_path: Path) -> None:
    if df.empty:
        return
    summary = (
        df.groupby(["model", "threshold"], as_index=False)
        .agg(
            mean_page_pct=("predicted_page_pct", "mean"),
            median_page_pct=("predicted_page_pct", "median"),
            mean_risk=("risk_index", "mean"),
        )
    )
    plt.figure(figsize=(10, 6))
    for model, group in summary.groupby("model"):
        plt.plot(
            group["threshold"],
            group["mean_page_pct"],
            marker="o",
            label=model,
        )
    plt.xlabel("Confidence threshold")
    plt.ylabel("Mean predicted page area (%)")
    plt.title("Prediction coverage by confidence threshold")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def run_optional_validation(
    model: YOLO,
    model_name: str,
    dataset_yaml: Optional[Path],
    imgsz: int,
    device: str,
    output_dir: Path,
) -> Optional[dict]:
    if dataset_yaml is None:
        return None
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    metrics = model.val(
        data=str(dataset_yaml),
        split="val",
        imgsz=imgsz,
        device=device,
        project=str(output_dir),
        name=f"{safe_name(model_name)}_validation",
        exist_ok=True,
    )
    return {key: float(value) for key, value in metrics.results_dict.items()}


def evaluate_group(
    group_name: str,
    paths: Sequence[Path],
    models: Dict[str, YOLO],
    thresholds: Sequence[float],
    args: argparse.Namespace,
    run_dir: Path,
) -> Tuple[List[dict], List[dict]]:
    page_records: List[dict] = []
    instance_records: List[dict] = []
    for path in paths:
        gray = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if gray is None:
            print(f"WARNING: unreadable image: {path}")
            continue
        variants = image_variants(
            gray,
            args.normalisation,
            args.background_blur_sigma,
            args.clahe_clip_limit,
        )
        for model_name, model in models.items():
            for variant_name, variant_image in variants.items():
                for threshold in thresholds:
                    output_dir = (
                        run_dir
                        / "predictions"
                        / safe_name(group_name)
                        / safe_name(model_name)
                        / safe_name(variant_name)
                        / f"conf_{threshold:.2f}"
                    )
                    output_dir.mkdir(parents=True, exist_ok=True)
                    output_path = output_dir / f"{safe_name(path.stem)}_prediction.png"
                    prediction, confidence_map, instances = tiled_predict(
                        model,
                        variant_image,
                        threshold,
                        args.tile_size,
                        args.tile_stride,
                        args.imgsz,
                        args.device,
                        args.iou,
                        args.minimum_output_component_area,
                    )
                    cv2.imwrite(str(output_path), prediction)
                    record = summarise_prediction(
                        path.name,
                        group_name,
                        model_name,
                        variant_name,
                        threshold,
                        variant_image,
                        confidence_map,
                        instances,
                        output_path,
                    )
                    page_records.append(record)
                    for instance in instances:
                        instance_records.append(
                            {
                                "group": group_name,
                                "page": path.name,
                                "model": model_name,
                                "variant": variant_name,
                                "threshold": threshold,
                                **instance,
                            }
                        )
                    print(
                        f"{group_name} | {path.name} | {model_name} | "
                        f"{variant_name} | {threshold:.2f}: "
                        f"page={record['predicted_page_pct']:.3f}% "
                        f"instances={record['instances']} "
                        f"risk={record['risk_index']:.3f}"
                    )
    return page_records, instance_records


def build_model_comparison(page_df: pd.DataFrame) -> pd.DataFrame:
    if page_df.empty or page_df["model"].nunique() < 2:
        return pd.DataFrame()
    index_columns = ["group", "page", "variant", "threshold"]
    pivot = page_df.pivot_table(
        index=index_columns,
        columns="model",
        values=["predicted_page_pct", "risk_index", "instances"],
        aggfunc="first",
    )
    pivot.columns = [f"{metric}__{model}" for metric, model in pivot.columns]
    return pivot.reset_index()


def write_recommendations(
    gst_df: pd.DataFrame,
    negative_df: pd.DataFrame,
    comparison_df: pd.DataFrame,
) -> List[str]:
    recommendations: List[str] = []
    if not negative_df.empty:
        worst = negative_df.sort_values("predicted_page_pct", ascending=False).iloc[0]
        recommendations.append(
            f"Highest hard-negative false-positive page: {worst['page']} "
            f"({worst['predicted_page_pct']:.3f}% predicted at confidence "
            f"{worst['threshold']:.2f}, model {worst['model']})."
        )
        grouped = negative_df.groupby(["model", "threshold"])["predicted_page_pct"].mean()
        best_index = grouped.idxmin()
        recommendations.append(
            f"Lowest mean hard-negative page coverage: model {best_index[0]} "
            f"at confidence {best_index[1]:.2f} ({grouped.loc[best_index]:.3f}%)."
        )
    if not gst_df.empty:
        grouped = gst_df.groupby(["model", "threshold"])["risk_index"].mean()
        best_index = grouped.idxmin()
        recommendations.append(
            f"Lowest mean GST risk index: model {best_index[0]} at confidence "
            f"{best_index[1]:.2f} ({grouped.loc[best_index]:.3f})."
        )
    if not comparison_df.empty:
        recommendations.append(
            "Use the model-comparison CSV to confirm whether reduced false-positive "
            "coverage is accompanied by acceptable visual handwriting retention."
        )
    recommendations.append(
        "Do not choose a threshold using predicted area alone; inspect the saved "
        "contact sheets because the GST pages currently lack complete pixel ground truth."
    )
    return recommendations

## Pre-evaluation safety gate

The notebook stops before GPU inference if checkpoints or required image folders are missing.

In [ ]:
for model_name,model_path in MODEL_PATHS.items():
    if not Path(model_path).exists():
        raise RuntimeError(f'Model not found for {model_name}: {model_path}')
if not GST_DIR.exists(): raise RuntimeError(f'GST directory missing: {GST_DIR}')
if not HARD_NEGATIVE_DIR.exists(): raise RuntimeError(f'Hard-negative directory missing: {HARD_NEGATIVE_DIR}')
if RUN_VALIDATION_METRICS and not DATASET_YAML.exists(): raise RuntimeError(f'Dataset YAML missing: {DATASET_YAML}')
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > T4 GPU, then Run all again.')
if TILE_STRIDE<=0 or TILE_STRIDE>TILE_SIZE: raise RuntimeError('TILE_STRIDE must be positive and no larger than TILE_SIZE.')
for threshold in CONFIDENCE_THRESHOLDS:
    if not 0<threshold<=1: raise RuntimeError(f'Invalid confidence threshold: {threshold}')

GST_FILES=image_files(GST_DIR)
NEGATIVE_FILES=image_files(HARD_NEGATIVE_DIR)
if not GST_FILES: raise RuntimeError('No GST images found.')
if not NEGATIVE_FILES: raise RuntimeError('No hard-negative test images found.')

RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR=OUTPUT_ROOT/RUN_NAME
TABLES_DIR=RUN_DIR/'tables'; CHARTS_DIR=RUN_DIR/'charts'; CONTACTS_DIR=RUN_DIR/'contact_sheets'
for folder in [RUN_DIR,TABLES_DIR,CHARTS_DIR,CONTACTS_DIR]: folder.mkdir(parents=True,exist_ok=True)

print('GPU:',torch.cuda.get_device_name(0))
print('GST pages:',len(GST_FILES))
print('Hard-negative pages:',len(NEGATIVE_FILES))
print('Output:',RUN_DIR)
print('SAFETY GATE PASSED')

## Load checkpoints and optionally calculate validation metrics

In [ ]:
MODELS={name:YOLO(str(path)) for name,path in MODEL_PATHS.items()}
validation_metrics={}
if RUN_VALIDATION_METRICS:
    for model_name,model in MODELS.items():
        metrics=run_optional_validation(
            model=model,
            model_name=model_name,
            dataset_yaml=DATASET_YAML,
            imgsz=MODEL_IMAGE_SIZE,
            device=DEVICE,
            output_dir=RUN_DIR/'validation',
        )
        if metrics is not None: validation_metrics[model_name]=metrics
print(json.dumps(validation_metrics,indent=2))

## Evaluate GST and hard-negative pages

This is the main inference step. Every page is processed at every configured confidence threshold for every selected model and normalisation variant.

In [ ]:
from types import SimpleNamespace
ARGS=SimpleNamespace(
    normalisation=NORMALISATION,
    background_blur_sigma=BACKGROUND_BLUR_SIGMA,
    clahe_clip_limit=CLAHE_CLIP_LIMIT,
    tile_size=TILE_SIZE,
    tile_stride=TILE_STRIDE,
    imgsz=MODEL_IMAGE_SIZE,
    device=DEVICE,
    iou=IOU_THRESHOLD,
    minimum_output_component_area=MINIMUM_OUTPUT_COMPONENT_AREA,
)

gst_records,gst_instances=evaluate_group(
    'gst',GST_FILES,MODELS,CONFIDENCE_THRESHOLDS,ARGS,RUN_DIR
)
negative_records,negative_instances=evaluate_group(
    'hard_negative',NEGATIVE_FILES,MODELS,CONFIDENCE_THRESHOLDS,ARGS,RUN_DIR
)

page_df=pd.DataFrame(gst_records+negative_records)
instance_df=pd.DataFrame(gst_instances+negative_instances)
gst_df=page_df[page_df['group']=='gst'].copy()
negative_df=page_df[page_df['group']=='hard_negative'].copy()
comparison_df=build_model_comparison(page_df)

print('Page-level rows:',len(page_df))
print('Instance-level rows:',len(instance_df))

## Export tables, charts and contact sheets

In [ ]:
page_df.to_csv(TABLES_DIR/'page_threshold_results.csv',index=False)
instance_df.to_csv(TABLES_DIR/'instance_predictions.csv',index=False)
gst_df.to_csv(TABLES_DIR/'gst_threshold_sweep.csv',index=False)

negative_leaderboard=negative_df.sort_values(
    ['threshold','predicted_page_pct'],ascending=[True,False]
)
negative_leaderboard.to_csv(
    TABLES_DIR/'hard_negative_false_positive_leaderboard.csv',index=False
)
comparison_df.to_csv(TABLES_DIR/'model_comparison.csv',index=False)
save_threshold_chart(page_df,CHARTS_DIR/'threshold_coverage_chart.png')

for (model_name,variant),subset in gst_df.groupby(['model','variant']):
    save_contact_sheet(
        subset,
        f'GST threshold sweep: {model_name} / {variant}',
        CONTACTS_DIR/f'gst_{safe_name(model_name)}_{safe_name(variant)}.png',
        MAXIMUM_CONTACT_SHEET_PAGES,
    )

for (model_name,variant),subset in negative_df.groupby(['model','variant']):
    worst_pages=(
        subset.groupby('page')['predicted_page_pct'].max()
        .sort_values(ascending=False)
        .head(MAXIMUM_CONTACT_SHEET_PAGES)
        .index
    )
    save_contact_sheet(
        subset[subset['page'].isin(worst_pages)],
        f'Hard-negative threshold sweep: {model_name} / {variant}',
        CONTACTS_DIR/f'hard_negative_{safe_name(model_name)}_{safe_name(variant)}.png',
        MAXIMUM_CONTACT_SHEET_PAGES,
    )

print('Tables:',TABLES_DIR)
print('Charts:',CHARTS_DIR)
print('Contact sheets:',CONTACTS_DIR)

## Display key results

In [ ]:
display_columns=[
    'page','model','variant','threshold','predicted_page_pct',
    'instances','median_instance_confidence','largest_instance_tile_pct','risk_index'
]

print('Worst hard-negative results:')
display(negative_leaderboard[display_columns].head(30))

print('GST threshold results:')
display(gst_df[display_columns].sort_values(['page','model','variant','threshold']))

contact_files=sorted(CONTACTS_DIR.glob('*.png'))
for path in contact_files:
    image=cv2.imread(str(path))
    plt.figure(figsize=(22,14))
    plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
    plt.title(path.name)
    plt.axis('off')
    plt.show()

chart_path=CHARTS_DIR/'threshold_coverage_chart.png'
if chart_path.exists():
    image=cv2.imread(str(chart_path))
    plt.figure(figsize=(12,7))
    plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

## Save recommendations and evaluation summary

In [ ]:
recommendations=write_recommendations(gst_df,negative_df,comparison_df)
summary={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'configuration':{
        'models':{name:str(path) for name,path in MODEL_PATHS.items()},
        'gst_dir':str(GST_DIR),
        'hard_negative_dir':str(HARD_NEGATIVE_DIR),
        'thresholds':CONFIDENCE_THRESHOLDS,
        'tile_size':TILE_SIZE,
        'tile_stride':TILE_STRIDE,
        'imgsz':MODEL_IMAGE_SIZE,
        'device':DEVICE,
        'iou':IOU_THRESHOLD,
        'minimum_output_component_area':MINIMUM_OUTPUT_COMPONENT_AREA,
        'normalisation':NORMALISATION,
    },
    'validation_metrics':validation_metrics,
    'gst_pages':len(GST_FILES),
    'hard_negative_pages':len(NEGATIVE_FILES),
    'recommendations':recommendations,
    'output_tables':{
        'page_results':str(TABLES_DIR/'page_threshold_results.csv'),
        'instance_results':str(TABLES_DIR/'instance_predictions.csv'),
        'gst_sweep':str(TABLES_DIR/'gst_threshold_sweep.csv'),
        'hard_negative_leaderboard':str(TABLES_DIR/'hard_negative_false_positive_leaderboard.csv'),
        'model_comparison':str(TABLES_DIR/'model_comparison.csv'),
    },
}
(RUN_DIR/'evaluation_summary.json').write_text(json.dumps(summary,indent=2))
(RUN_DIR/'recommendations.txt').write_text('\n'.join(f'- {item}' for item in recommendations))

print('EVALUATION COMPLETE')
print('Summary:',RUN_DIR/'evaluation_summary.json')
print('Recommendations:')
for item in recommendations: print('-',item)